# Inferencia para vLLM (solo funciona para GPU)

In [1]:
!pip install vllm

In [2]:
from vllm import LLM, SamplingParams
from datasets import load_dataset
from tqdm import tqdm
import pandas as pd
import numpy as np
import psutil
import torch
import time


# vLLM (75)

### Configuración Global

In [3]:
print("Cargando dataset Dolly 15k...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
SAMPLE_SIZE = 75
test_subset = dataset.shuffle(seed=42).select(range(SAMPLE_SIZE))

# --- PARÁMETROS ---
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
PROMPTS = [f"Instruction: {row['instruction']}\nResponse:" for row in test_subset]

# Configuración de generación idéntica para todos
GEN_CONFIG = {
    "max_new_tokens": 128,
    "temperature": 0.7,
    "top_k": 50,
    "do_sample": True
}

# Tabla para guardar resultados
results = []

# Determine if CUDA is available, otherwise use CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.float16
    print("CUDA está disponible. Usando GPU.")


def get_vram_usage():
    """Devuelve la memoria VRAM usada en GB, para vLLM tampoco se puede usar CPU"""
    if DEVICE == "cuda":
        return torch.cuda.get_device_properties(0).total_memory * 0.6 / (1024**3)


def clear_cache():
    """Limpia la memoria GPU entre pruebas, o no hace nada si es CPU"""
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats() # Only call if CUDA is available
    import gc
    gc.collect()

print(f"Modelo: {MODEL_ID}")
print(f"Número de pruebas (Prompts): {len(PROMPTS)}")
print(f"Ejemplo de prompt:\n{PROMPTS[0][:100]}...")

Cargando dataset Dolly 15k...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

CUDA está disponible. Usando GPU.
Modelo: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Número de pruebas (Prompts): 75
Ejemplo de prompt:
Instruction: Who were the children of the legendary Garth Greenhand, the High King of the First Men ...


In [4]:
def benchmark_vllm():
    print("\n--- Iniciando Benchmark: vLLM ---")

    # Configuración de parámetros
    sampling_params = SamplingParams(
        temperature=GEN_CONFIG.get("temperature", 0.7),
        max_tokens=GEN_CONFIG.get("max_new_tokens", 128),
        top_k=GEN_CONFIG.get("top_k", 50)
    )

    # Cargar vLLM
    llm = LLM(
        model=MODEL_ID,
        dtype="float16",
        gpu_memory_utilization=0.6, # Usa un 60% de la memoria, en Colab T4 (16GB)
        tensor_parallel_size=1
    )
    # Warmup
    llm.generate(["Warmup: Hello world"], sampling_params)

    # Generación, no hay loop poruqe se hace en paralelo
    print(f"Procesando {len(PROMPTS)} prompts en paralelo...")

    start_time = time.time()
    outputs = llm.generate(PROMPTS, sampling_params)
    end_time = time.time()

    # Cálculo de Métricas
    total_duration = end_time - start_time
    total_tokens = sum([len(o.outputs[0].token_ids) for o in outputs])

    avg_latency = total_duration / len(PROMPTS)
    throughput = total_tokens / total_duration

    # Medir memoria pico
    # vLLM reserva la memoria al inicio, así que medimos lo que ha reservado Torch
    peak_mem = get_vram_usage()

    # Guardar resultados
    results.append({
        "Framework": "vLLM",
        "Avg Latency (s)": avg_latency,
        "Throughput (tok/s)": throughput,
        "Peak Memory (GB)": peak_mem
    })

    print(f"vLLM completado.")

    # Limpieza final
    import gc
    del llm
    gc.collect()
    torch.cuda.empty_cache()

benchmark_vllm()


--- Iniciando Benchmark: vLLM ---
INFO 12-29 00:35:27 [utils.py:253] non-default args: {'dtype': 'float16', 'gpu_memory_utilization': 0.6, 'disable_log_stats': True, 'model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'}


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

INFO 12-29 00:36:02 [model.py:514] Resolved architecture: LlamaForCausalLM
WARNING 12-29 00:36:02 [model.py:2005] Casting torch.bfloat16 to torch.float16.
INFO 12-29 00:36:02 [model.py:1661] Using max model len 2048
INFO 12-29 00:36:06 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Procesando 75 prompts en paralelo...


Adding requests:   0%|          | 0/75 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/75 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

vLLM completado.


In [5]:
results

[{'Framework': 'vLLM',
  'Avg Latency (s)': 0.03337246576944987,
  'Throughput (tok/s)': 2785.52986291766,
  'Peak Memory (GB)': 8.84476318359375}]

##### El consumo de GB daba 0.0 (con la función que usan los distintos frameworks), eso es porque vLLM lo gestiona de forma diferente, sin embargo al principio se ha configurado gpu_memory_utilization=0.6, el consumo real es el 60% de la tarjeta. Es decir, ~9.0 GB (por usar la T4 de Colab que tiene 16GB)

# vLLM (1)

## Configuración global

In [6]:
print("Cargando dataset Dolly 15k...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
SAMPLE_SIZE = 1
test_subset = dataset.shuffle(seed=42).select(range(SAMPLE_SIZE))

# --- PARÁMETROS ---
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
PROMPTS = [f"Instruction: {row['instruction']}\nResponse:" for row in test_subset]

# Configuración de generación idéntica para todos
GEN_CONFIG = {
    "max_new_tokens": 128,
    "temperature": 0.7,
    "top_k": 50,
    "do_sample": True
}

# Tabla para guardar resultados
results = []

# Determine if CUDA is available, otherwise use CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.float16
    print("CUDA está disponible. Usando GPU.")


def get_vram_usage():
    """Devuelve la memoria VRAM usada en GB, para vLLM tampoco se puede usar CPU"""
    if DEVICE == "cuda":
        return torch.cuda.get_device_properties(0).total_memory * 0.6 / (1024**3)


def clear_cache():
    """Limpia la memoria GPU entre pruebas, o no hace nada si es CPU"""
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats() # Only call if CUDA is available
    import gc
    gc.collect()

print(f"Modelo: {MODEL_ID}")
print(f"Número de pruebas (Prompts): {len(PROMPTS)}")
print(f"Ejemplo de prompt:\n{PROMPTS[0][:100]}...")

Cargando dataset Dolly 15k...
CUDA está disponible. Usando GPU.
Modelo: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Número de pruebas (Prompts): 1
Ejemplo de prompt:
Instruction: Who were the children of the legendary Garth Greenhand, the High King of the First Men ...


In [7]:
def benchmark_vllm():
    print("\n--- Iniciando Benchmark: vLLM ---")

    # Configuración de parámetros
    sampling_params = SamplingParams(
        temperature=GEN_CONFIG.get("temperature", 0.7),
        max_tokens=GEN_CONFIG.get("max_new_tokens", 128),
        top_k=GEN_CONFIG.get("top_k", 50)
    )

    # Cargar vLLM
    llm = LLM(
        model=MODEL_ID,
        dtype="float16",
        gpu_memory_utilization=0.6, # Usa un 60% de la memoria, en Colab T4 (16GB)
        tensor_parallel_size=1
    )
    # Warmup
    llm.generate(["Warmup: Hello world"], sampling_params)

    # Generación, no hay loop poruqe se hace en paralelo
    print(f"Procesando {len(PROMPTS)} prompts en paralelo...")

    start_time = time.time()
    outputs = llm.generate(PROMPTS, sampling_params)
    end_time = time.time()

    # Cálculo de Métricas
    total_duration = end_time - start_time
    total_tokens = sum([len(o.outputs[0].token_ids) for o in outputs])

    avg_latency = total_duration / len(PROMPTS)
    throughput = total_tokens / total_duration

    # Medir memoria pico
    # vLLM reserva la memoria al inicio, así que medimos lo que ha reservado Torch
    peak_mem = get_vram_usage()

    # Guardar resultados
    results.append({
        "Framework": "vLLM",
        "Avg Latency (s)": avg_latency,
        "Throughput (tok/s)": throughput,
        "Peak Memory (GB)": peak_mem
    })

    print(f"vLLM completado.")

    # Limpieza final
    import gc
    del llm
    gc.collect()
    torch.cuda.empty_cache()

benchmark_vllm()


--- Iniciando Benchmark: vLLM ---
INFO 12-29 00:40:23 [utils.py:253] non-default args: {'dtype': 'float16', 'gpu_memory_utilization': 0.6, 'disable_log_stats': True, 'model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'}
INFO 12-29 00:40:24 [model.py:514] Resolved architecture: LlamaForCausalLM
WARNING 12-29 00:40:24 [model.py:2005] Casting torch.bfloat16 to torch.float16.
INFO 12-29 00:40:24 [model.py:1661] Using max model len 2048
INFO 12-29 00:40:24 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 12-29 00:41:14 [llm.py:360] Supported tasks: ['generate']


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Procesando 1 prompts en paralelo...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

vLLM completado.


In [8]:
results

[{'Framework': 'vLLM',
  'Avg Latency (s)': 1.4906797409057617,
  'Throughput (tok/s)': 85.86686763598537,
  'Peak Memory (GB)': 8.84476318359375}]